In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import files
uploaded = files.upload()  # upload your service account JSON

from google.cloud import bigquery
client = bigquery.Client.from_service_account_json(list(uploaded.keys())[0])
print("Connected to project:", client.project)

Saving phonic-chemist-509603-v9-6d0523a7b77d.json to phonic-chemist-509603-v9-6d0523a7b77d (1).json
Connected to project: phonic-chemist-509603-v9


In [3]:
import pandas as pd

sales = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/sales_train_validation.csv")
calendar = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/calendar.csv")
prices = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/sell_prices.csv")

print(sales.shape, calendar.shape, prices.shape)

(30490, 1919) (1969, 14) (6841121, 4)


In [4]:
client.delete_table("phonic-chemist-509603-v9.m5_analytics.mart_daily_sales", not_found_ok=True)
print("Cleared old mart_daily_sales if it existed")

Cleared old mart_daily_sales if it existed


In [5]:
print(dir())

['In', 'Out', '_', '__', '___', '__builtin__', '__builtins__', '__doc__', '__loader__', '__name__', '__package__', '__spec__', '_dh', '_i', '_i1', '_i2', '_i3', '_i4', '_i5', '_ih', '_ii', '_iii', '_oh', 'bigquery', 'calendar', 'client', 'drive', 'exit', 'files', 'get_ipython', 'pd', 'prices', 'quit', 'sales', 'uploaded']


In [2]:
print('weekly_mart' in dir())

False


In [1]:
%cd /content
!zip -r dbt_project.zip dbt_project -x "dbt_project/target/*" "dbt_project/logs/*" "dbt_project/dbt_packages/*"

from google.colab import files
files.download("dbt_project.zip")

/content
	zip warning: name not matched: dbt_project

zip error: Nothing to do! (dbt_project.zip)


FileNotFoundError: Cannot find file: dbt_project.zip

In [ ]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales.columns if c.startswith("d_")]

sales_long = sales.melt(id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="units_sold")
sales_long["day_num"] = sales_long["d"].str.replace("d_", "").astype(int)

calendar["day_num"] = range(1, len(calendar) + 1)
sales_long = sales_long.merge(calendar[["day_num", "date", "wm_yr_wk"]], on="day_num", how="left")

sales_long = sales_long.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
sales_long["revenue"] = sales_long["units_sold"] * sales_long["sell_price"]

weekly_mart = (
    sales_long.groupby(["store_id", "cat_id", "dept_id", "wm_yr_wk"])
    .agg(total_units_sold=("units_sold", "sum"),
         total_revenue=("revenue", "sum"),
         avg_sell_price=("sell_price", "mean"))
    .reset_index()
)
print(weekly_mart.shape)
print(weekly_mart.head())

In [ ]:
table_id = "phonic-chemist-509603-v9.m5_analytics.mart_weekly_sales"
job = client.load_table_from_dataframe(weekly_mart, table_id)
job.result()
print("Loaded mart_weekly_sales:", weekly_mart.shape)